# Dataset Exploration

## Objective

This notebook explores the xLAM function-calling dataset used for fine-tuning and evaluating a language model for tool calling.

The objectives are to:

- Understand the dataset structure
- Inspect queries, available tools, and expected tool calls
- Analyze the number of expected tool calls per example
- Analyze the number of available tools per example
- Understand the complexity of the function-calling task

In [ ]:
import json
import pandas as pd 
from collections import Counter
from datasets import loada_dataset
from huggingface_hub import login

In [ ]:
login()

dataset = load_dataset(
    "Salesforce/xlam-function-calling-60k"
)
print("Dataset structure:")
print(dataset)
print("\nColumn names:")
print(dataset["train"].column_names)
print("\nFirst example:")
print(dataset["train"][0])

example = dataset["train"][0]
parsed_answers = json.loads(example["answers"])
parsed_tools = json.loads(example["tools"])

print("Parsed Answers:")
print(parsed_answers)

print("\nParsed Tools:")
print(parsed_tools)

# Each dataset example contains:

- `query`: The user's request
- `tools`: The tools available to the model
- `answers`: The expected tool call or tool calls
- `id`: Unique example identifier

In [ ]:
expected_tool_call_counts = []
available_tool_counts = []

for example in dataset["train"]:

    answers = json.loads(example["answers"])
    tools = json.loads(example["tools"])

    expected_tool_call_counts.append(len(answers))
    available_tool_counts.append(len(tools))

expected_calls_distribution = (
    pd.Series(expected_tool_call_counts)
    .value_counts()
    .sort_index()
)

print(expected_calls_distribution)

available_tools_distribution = (
    pd.Series(available_tool_counts)
    .value_counts()
    .sort_index()
)

print(available_tools_distribution)

### Interpretation

The number of expected tool calls represents how many function calls
the correct answer contains.

For example:

One expected call:

[
    {"name": "tool_a", "arguments": {...}}
]

Two expected calls:

[
    {"name": "tool_a", "arguments": {...}},
    {"name": "tool_b", "arguments": {...}}
]

### Interpretation

The number of available tools represents how many different tool
definitions the model can choose from for a particular example.

The number of available tools and expected tool calls are independent.

For example:

Available tools = 8

Expected tool calls = 2

The model is given eight possible tools but only needs to select
and call two of them.

# Key Findings

- The dataset contains 60,000 function-calling examples.
- Each example contains a user query, available tool definitions,
  and expected tool calls.
- Examples vary in the number of available tools.
- Examples also vary significantly in the number of expected tool calls.
- The number of available tools and expected tool calls are independent.
- The dataset includes both simple single-tool-call examples and
  more complex multi-tool-call examples.

These findings will be used in the next notebook to create a
grouped stratified train, validation, and test split.